<a href="https://colab.research.google.com/github/rodrigoiyg27/TopicosEspeciales2026/blob/main/CasoPracticoClase4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
url= "https://raw.githubusercontent.com/rodrigoiyg27/TopicosEspeciales2026/main/Customer%20Call%20List.xlsx"
df = pd.read_excel(url, "Call List")
print("Dimensiones originales:", df.shape)
df.head()

Dimensiones originales: (21, 8)


,CustomerID,First_Name,Last_Name,Phone_Number,Address,Paying Customer,Do_Not_Contact,Not_Useful_Column
0,1001,Frodo,Baggins,123-545-5421,"123 Shire Lane, Shire",Yes,No,True
1,1002,Abed,Nadir,123/643/9775,93 West Main Street,No,Yes,False
2,1003,Walter,/White,7066950392,298 Drugs Driveway,N,NaN,True
3,1004,Dwight,Schrute,123-543-2345,"980 Paper Avenue, Pennsylvania, 18503",Yes,Y,True
4,1005,Jon,Snow,876|678|3469,123 Dragons Road,Y,No,True


Eliminar Columnas sin valor analitico

In [ ]:
df = df.drop(columns=["Not_Useful_Column"], errors="ignore")

Eliminar duplicados exactos

In [ ]:
duplicados = df.duplicated().sum()
print(f"Duplicados exactos encontrados: {duplicados}")
df = df.drop_duplicates()

Duplicados exactos encontrados: 1


Trim y normalización de nombres

In [ ]:
import numpy as np
for col in ["First_Name", "Last_Name"]:
    datos_nulos = df[col].isna()  # marca Gandalf como nulo, ANTES de tocar la columna
    df[col] = (
        df[col].astype(str)
               .str.strip()                 # quita espacios sobrantes
               .str.replace(r"[._/]+", "", regex=True)  # quita símbolos sueltos
               .str.title()                 # formato "Nombre Apellido"
    )
    df.loc[datos_nulos, col] = np.nan

Estandarizar telefono

In [ ]:
import re
import numpy as np
def limpiar_telefono(valor):
    digitos = re.sub(r"\D", "", str(valor))   # deja solo números
    if len(digitos) == 10:
        # Formato uniforme: 123-456-7890
        return f"{digitos[0:3]}-{digitos[3:6]}-{digitos[6:10]}"
    return np.nan  # teléfonos incompletos o inválidos quedan como nulo explícito

df["Phone_Number"] = df["Phone_Number"].apply(limpiar_telefono)

Separar dirección en componentes (calle, estado, zip)

In [ ]:
direccion_split = df["Address"].astype(str).str.split(",", n=2, expand=True)
df["Street_Address"] = direccion_split[0].str.strip()
df["State"] = direccion_split[1].str.strip() if 1 in direccion_split else np.nan
df["Zip_Code"] = direccion_split[2].str.strip() if 2 in direccion_split else np.nan
df = df.drop(columns=["Address"])

Unificar columnas booleanas con formato inconsistente

In [ ]:
mapa_booleano = {"yes": True, "y": True, "no": False, "n": False}

for col in ["Paying Customer", "Do_Not_Contact"]:
    df[col] = (
        df[col].astype(str).str.strip().str.lower()
               .replace({"nan": np.nan, "n/a": np.nan})
               .map(mapa_booleano)
    )

Quitar clientes que pidieron ser no contactados

In [ ]:
df = df[df["Do_Not_Contact"] != True]

Tratar nulos faltantes

In [ ]:
df["Phone_Number"] = df["Phone_Number"].fillna("Sin teléfono válido")

Verificacion final

In [ ]:
print("Dimensiones finales:", df.shape)
print(df.isna().sum())
df.to_csv("clientes_limpios.csv", index=False, encoding="utf-8-sig")

Dimensiones finales: (16, 9)
CustomerID          0
First_Name          0
Last_Name           1
Phone_Number        0
Paying Customer     0
Do_Not_Contact      4
Street_Address      0
State              11
Zip_Code           16
dtype: int64
